# APH/PPH split proportions artifact vs csv

## Setup

In [ ]:
import pandas as pd, numpy as np, os
from vivarium import Artifact
import db_queries
from get_draws.api import get_draws
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
from vivarium_gates_mncnh.constants import paths

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning) 

In [ ]:
# Parameters cell for papermill
model_dir = "model34.0"

In [ ]:
base_artifacts_dir = Path("/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/") / model_dir

In [ ]:
artifact_paths = {p.stem.title(): p for p in base_artifacts_dir.iterdir()}
artifact_paths

In [ ]:
locations = list(artifact_paths.keys())
locations

In [ ]:
location_ids = db_queries.get_ids('location')
location_ids = location_ids.loc[location_ids.location_name.str.lower().isin([x.lower() for x in locations])]
location_ids

In [ ]:
draws_we_simulate = [22, 60, 71, 79, 115, 118, 146, 167, 197, 244]
draws_we_simulate = ["draw_"+str(x) for x in draws_we_simulate]
draws_we_simulate

In [ ]:
def load_yaml_file(path):
    with open(path) as stream:
        return yaml.safe_load(stream)

In [ ]:
def read_artifact(key, filter_terms=['sex == Female' , 'age_start > 5', 'age_end < 60']):
    all_locations_data = []
    for location in locations:
        art = Artifact(artifact_paths[location], filter_terms=filter_terms)
        location_data = art.load(key)
        if not isinstance(location_data, pd.DataFrame):
            location_data = pd.DataFrame({'value': location_data, 'location': location}, index=[0]).set_index('location')
        else:
            location_data['location'] = location
            location_data = location_data.reset_index().set_index(['location'] + [c for c in location_data.index.names if c is not None])
        all_locations_data.append(location_data)

    all_locations_data = pd.concat(all_locations_data)
    #if 'draw' in all_locations_data.columns[0]:
    #    all_locations_data = all_locations_data[[f'draw_{draw}' for draw in draws]]
    #else:
    #    for draw in draws:
    #        all_locations_data[f'draw_{draw}'] = all_locations_data['value']
    #    all_locations_data = all_locations_data.drop(columns='value')
    return all_locations_data

In [ ]:
pp_frac = read_artifact('cause.maternal_hemorrhage.postpartum_fraction')[draws_we_simulate]
pp_frac

In [ ]:
def describe_rowwise(df, percentiles=(0.025, 0.975)):
    # The pandas .describe() method describes columns
    # We can transpose before and after to describe rows instead
    return df.transpose().describe(percentiles=percentiles).transpose()

In [ ]:
art = describe_rowwise(np.log(pp_frac))[["mean", "std"]].reset_index()
art = art[art.location == "Nigeria"].drop(columns=["location", "year_start", "year_end"]).set_index(["age_start", "age_end", "sex"]).reset_index()
art


In [ ]:
csv = pd.read_csv(paths.PPH_CROSSWALK_PARAMETERS_CSV)
csv = csv.set_index(["age_start", "age_end", "sex"])
csv = csv.rename(columns={"pred_diff_mean": "mean", "pred_diff_sd": "std"}).reset_index()
csv

In [ ]:
age_starts = art.age_start.unique()

for age in age_starts:
    art_for_age = art[art.age_start == age]
    csv_for_age = csv[csv.age_start == age]
    plt.scatter(art_for_age["mean"], art_for_age["mean"] / csv_for_age["mean"], label=age)
plt.axhline(1, color='k', linestyle='--')
plt.axhline(1.1, color='k', linestyle='dotted')
plt.axhline(0.9, color='k', linestyle='dotted')
plt.xlabel(f'Artifact')
plt.ylabel('Artifact / CSV')
plt.title('Relative error, postpartum fraction mean')
plt.legend(title='Age Group Start', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

for age in age_starts:
    art_for_age = art[art.age_start == age]
    csv_for_age = csv[csv.age_start == age]
    plt.scatter(art_for_age["std"], art_for_age["std"] / csv_for_age["std"], label=age)
plt.axhline(1, color='k', linestyle='--')
plt.axhline(1.1, color='k', linestyle='dotted')
plt.axhline(0.9, color='k', linestyle='dotted')
#plt.ylim((art["mean"] / csv["mean"]).min(), (art["mean"] / csv["mean"]).max())
plt.xlabel(f'Artifact')
plt.ylabel('Artifact / CSV')
plt.title('Relative error, postpartum fraction standard deviation')
plt.legend(title='Age Group Start', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# Note these values are surprisingly variable. We believe this is because the uncertainty is so large and we are only sampling 10 draws.